# 10-02 Agent 评估体系

Agent 不同于传统软件，其输出非确定性、多步骤、主观质量难以量化。本节构建一套实用的评估框架。

**本节目标**：
- 理解 Agent 评估的核心难点
- 实现 LLM-as-Judge 自动评分
- 实现 Golden Dataset 对比评估
- 计算任务完成率、工具使用准确率等指标

---

### 为什么 Agent 评估很难？

| 难点 | 说明 | 传统软件对比 |
|------|------|--------------|
| **非确定性** | 同一输入可能产生不同输出 | 传统软件输出确定 |
| **多步骤链路** | 错误可能在任意步骤引入 | 单元测试可覆盖 |
| **主观质量** | "好文案"没有唯一标准答案 | 有明确的对错 |
| **工具交互** | 需要评估工具选择和参数是否正确 | 无外部工具依赖 |
| **上下文敏感** | 评估需考虑完整对话历史 | 通常无状态 |

In [ ]:
import sys, json
sys.path.insert(0, "..")
from dotenv import load_dotenv
load_dotenv("../.env")

from utils.llm_client import call_llm

# ========== 1. LLM-as-Judge：用 LLM 给 Agent 输出打分 ==========

JUDGE_PROMPT = """你是一个专业的广告文案评审员。请对以下B站广告文案从三个维度打分（1-5分）：

**任务要求**: {task}
**Agent 输出**: {output}

请严格按以下 JSON 格式返回评分：
{{
  "correctness": {{"score": <1-5>, "reason": "<理由>"}},
  "helpfulness": {{"score": <1-5>, "reason": "<理由>"}},
  "safety": {{"score": <1-5>, "reason": "<理由>"}}
}}

评分标准：
- correctness（正确性）：是否准确完成任务要求
- helpfulness（有用性）：文案质量、创意度、是否有商业价值
- safety（安全性）：是否有违规、敏感或误导内容
"""


def llm_judge(task: str, agent_output: str) -> dict:
    """用 LLM 作为裁判，对 Agent 输出进行多维度评分。"""
    prompt = JUDGE_PROMPT.format(task=task, output=agent_output)
    # 实际调用 LLM 评分（需要 API Key）
    # response = call_llm(prompt, system="你是严格的评分专家，只输出JSON。")
    # 模拟返回结果
    return {
        "correctness": {"score": 4, "reason": "文案符合B站风格，但缺少具体优惠信息"},
        "helpfulness": {"score": 5, "reason": "创意新颖，融合了ACG元素，商业价值高"},
        "safety": {"score": 5, "reason": "内容健康，无违规风险"}
    }


# ---- 演示 ----
task = "为B站游戏联运活动写一条15字以内的广告标题"
outputs = [
    "开黑不孤单，B站联运新游首发！",
    "超好玩的游戏来了",
    "限时福利！B站专属游戏礼包等你领",
]

print("=== LLM-as-Judge 评估 ===")
for i, output in enumerate(outputs):
    scores = llm_judge(task, output)
    avg = sum(d["score"] for d in scores.values()) / len(scores)
    print(f"\n文案{i+1}: '{output}'")
    for dim, detail in scores.items():
        print(f"  {dim}: {detail['score']}/5 - {detail['reason']}")
    print(f"  综合得分: {avg:.1f}/5")

In [ ]:
# ========== 2. Golden Dataset 评估：与参考答案对比 ==========

GOLDEN_DATASET = [
    {
        "task": "为B站美妆品牌写广告标题",
        "reference": "美妆好物，UP主亲测推荐",
        "agent_output": "UP主同款美妆，限时特惠中",
    },
    {
        "task": "为B站教育课程写广告标题",
        "reference": "名师带你学，B站独家课程",
        "agent_output": "B站名师课堂，高效提分秘籍",
    },
    {
        "task": "为B站3C产品写广告标题",
        "reference": "极客必入，B站首发评测",
        "agent_output": "新品首发，科技宅的最爱",
    },
    {
        "task": "为B站食品品牌写广告标题",
        "reference": "干饭人集合！B站吃货节来了",
        "agent_output": "好吃不贵，宅家必备零食",
    },
]


def keyword_overlap_score(reference: str, output: str) -> float:
    """简单的关键词重叠率评估（教学用，实际应使用 BLEU/ROUGE/BERTScore）。"""
    ref_chars = set(reference)
    out_chars = set(output)
    overlap = ref_chars & out_chars
    precision = len(overlap) / len(out_chars) if out_chars else 0
    recall = len(overlap) / len(ref_chars) if ref_chars else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
    return round(f1, 3)


def evaluate_golden_dataset(dataset: list[dict]) -> dict:
    """在 Golden Dataset 上评估 Agent 的整体表现。"""
    scores = []
    print("=== Golden Dataset 评估 ===")
    for item in dataset:
        score = keyword_overlap_score(item["reference"], item["agent_output"])
        scores.append(score)
        print(f"  任务: {item['task']}")
        print(f"  参考: {item['reference']}")
        print(f"  输出: {item['agent_output']}")
        print(f"  F1得分: {score:.3f}\n")

    avg_score = sum(scores) / len(scores)
    print(f"平均 F1 得分: {avg_score:.3f}")
    print(f"及格率（>0.3）: {sum(1 for s in scores if s > 0.3)}/{len(scores)}")
    return {"avg_f1": avg_score, "scores": scores}


results = evaluate_golden_dataset(GOLDEN_DATASET)

In [ ]:
# ========== 3. 任务完成率 & 工具使用准确率 ==========

def evaluate_task_metrics(agent_runs: list[dict]) -> dict:
    """评估 Agent 多次运行的任务完成率和工具使用准确率。

    agent_runs: [{"task_completed": bool, "tools_expected": [...], "tools_used": [...]}]
    """
    total = len(agent_runs)
    completed = sum(1 for r in agent_runs if r["task_completed"])
    completion_rate = completed / total if total else 0

    # 工具使用准确率：预期工具 vs 实际使用工具
    tool_correct = 0
    tool_total = 0
    for run in agent_runs:
        expected = set(run["tools_expected"])
        used = set(run["tools_used"])
        if expected:
            tool_total += 1
            if used == expected:
                tool_correct += 1

    tool_accuracy = tool_correct / tool_total if tool_total else 0

    return {
        "task_completion_rate": round(completion_rate, 3),
        "tool_usage_accuracy": round(tool_accuracy, 3),
        "total_runs": total,
        "completed": completed,
    }


# ---- 模拟 B站广告 Agent 的多次运行记录 ----
agent_runs = [
    {"task_completed": True,  "tools_expected": ["search_trends", "generate_copy"], "tools_used": ["search_trends", "generate_copy"]},
    {"task_completed": True,  "tools_expected": ["search_trends", "generate_copy"], "tools_used": ["generate_copy", "search_trends"]},
    {"task_completed": False, "tools_expected": ["search_trends", "generate_copy"], "tools_used": ["generate_copy"]},
    {"task_completed": True,  "tools_expected": ["analyze_audience"],              "tools_used": ["analyze_audience"]},
    {"task_completed": True,  "tools_expected": ["analyze_audience", "ab_test"],   "tools_used": ["analyze_audience", "ab_test"]},
    {"task_completed": False, "tools_expected": ["generate_copy"],                 "tools_used": []},
    {"task_completed": True,  "tools_expected": ["search_trends"],                 "tools_used": ["search_trends"]},
    {"task_completed": True,  "tools_expected": ["generate_copy", "review_copy"],  "tools_used": ["generate_copy", "review_copy"]},
]

metrics = evaluate_task_metrics(agent_runs)
print("=== Agent 运行指标 ===")
print(f"  总运行次数: {metrics['total_runs']}")
print(f"  任务完成数: {metrics['completed']}")
print(f"  任务完成率: {metrics['task_completion_rate']:.1%}")
print(f"  工具使用准确率: {metrics['tool_usage_accuracy']:.1%}")

## RAGAS 评估指标（RAG 场景）

当 Agent 包含 RAG 组件时，可使用 RAGAS 框架进行专项评估：

| 指标 | 含义 | 计算方式 |
|------|------|----------|
| **Faithfulness** | 答案是否忠于检索到的上下文 | LLM 判断答案中的声明是否能从上下文推出 |
| **Answer Relevancy** | 答案与问题的相关度 | 用 LLM 从答案反向生成问题，计算与原问题的相似度 |
| **Context Precision** | 检索的上下文中相关文档排名是否靠前 | 相关文档的加权排名得分 |
| **Context Recall** | 检索到的上下文是否覆盖了参考答案的关键信息 | 参考答案中能被上下文支持的比例 |

```python
# RAGAS 使用示例（需 pip install ragas）
# from ragas import evaluate
# from ragas.metrics import faithfulness, answer_relevancy
# result = evaluate(dataset, metrics=[faithfulness, answer_relevancy])
```

## 面试速记

| 问题 | 要点 |
|------|------|
| 如何评估 Agent 输出质量 | LLM-as-Judge 多维度打分（正确性/有用性/安全性），成本低、可扩展 |
| Golden Dataset 的作用 | 回归测试基准线，防止模型升级导致质量下降 |
| 工具使用准确率为什么重要 | Agent 选错工具会导致整条链路失败，是可靠性的关键指标 |
| RAGAS vs 通用评估 | RAGAS 专注 RAG 场景的检索+生成质量，通用评估更关注端到端任务完成 |
| 评估的最大挑战 | 主观质量难量化，建议结合自动评估 + 人工抽检 |

**下一节**: `03_agent_monitoring.ipynb` — Agent 监控与追踪